# Stress Prediction — Tree 하이퍼파라미터 Grid Search

오늘 유일하게 안 건드린 부분: `min_samples_leaf`, `max_features`, `n_estimators`.
지금까지 이 값들은 v34 원본 그대로(1, 1, 1200) 고정하고 quantile/weight만 바꿨습니다.

**주의**: 이건 quantile grid search와 달리 조합마다 모델을 처음부터 다시 학습해야 해서
느립니다. 조합 수를 최소화했고(3x3x2=18), 트리 개수도 탐색용으로 줄였습니다.
시간이 없으면 `MAX_FEATURES_GRID`와 `MIN_SAMPLES_LEAF_GRID`를 각각 2개로 줄이세요.


In [1]:

from __future__ import annotations

import os
import time
from itertools import combinations, product
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============== 탐색 그리드 (시간 없으면 각 리스트를 2개로 줄이세요) ==============
N_ESTIMATORS_GRID = [600]              # 탐색은 600으로 고정(속도), 최종에만 1200으로 재확인
MIN_SAMPLES_LEAF_GRID = [1, 2, 3]
MAX_FEATURES_GRID = [1, 2, 3]
# ===================================================================

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target, n_estimators, min_samples_leaf, max_features) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        criterion="squared_error",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


## Grid Search 실행 (pair는 조합마다 매번 다시 계산할 필요 없음 -> fold당 한 번만)

In [2]:

train = pd.read_csv(DATA_DIR / "train.csv")
raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
grid_combos = list(product(N_ESTIMATORS_GRID, MIN_SAMPLES_LEAF_GRID, MAX_FEATURES_GRID))
print(f"탐색할 조합 수: {len(grid_combos)}, fold 수: 5 -> 총 {len(grid_combos) * 5}번 트리 학습")

scores = {combo: [] for combo in grid_combos}
start = time.time()

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].to_numpy(dtype=float)

    # pair는 하이퍼파라미터와 무관하므로 fold당 한 번만 계산
    pred_pair = pair_prediction(train_feat, val_feat, y_train)

    for n_est, min_leaf, max_feat in grid_combos:
        pred_tree = tree_prediction(train_feat, val_feat, y_train, n_est, min_leaf, max_feat)
        combined = np.clip(np.round((1 - PAIR_WEIGHT) * pred_tree + PAIR_WEIGHT * pred_pair, 2), 0.0, 1.0)
        scores[(n_est, min_leaf, max_feat)].append(mean_absolute_error(y_val, combined))

    elapsed = time.time() - start
    print(f"Fold {fold} done ({elapsed:.0f}s 경과)")

results = pd.DataFrame([
    {"n_estimators": n, "min_samples_leaf": m, "max_features": f, "mean_MAE": np.mean(v)}
    for (n, m, f), v in scores.items()
]).sort_values("mean_MAE").reset_index(drop=True)

print()
print("(참고) v34 원본(min_samples_leaf=1, max_features=1) CV MAE: 0.148033")
print()
display(results)


탐색할 조합 수: 9, fold 수: 5 -> 총 45번 트리 학습
Fold 1 done (7s 경과)
Fold 2 done (14s 경과)
Fold 3 done (20s 경과)
Fold 4 done (27s 경과)
Fold 5 done (34s 경과)

(참고) v34 원본(min_samples_leaf=1, max_features=1) CV MAE: 0.148033



,n_estimators,min_samples_leaf,max_features,mean_MAE
0,600,1,1,0.148340
1,600,1,3,0.148507
2,600,1,2,0.148547
3,600,2,3,0.192350
4,600,2,2,0.204153
5,600,3,3,0.204573
6,600,3,2,0.212833
7,600,2,1,0.218547
8,600,3,1,0.220487


## 최적 조합으로 최종 제출 파일 생성 (n_estimators는 1200으로 복원)

위 결과에서 `mean_MAE`가 0.148033보다 낮으면 `BEST_MIN_LEAF`, `BEST_MAX_FEATURES`를 그 조합으로 수정하세요.


In [3]:

BEST_MIN_LEAF = 1        # grid 결과 1위로 수정
BEST_MAX_FEATURES = 1    # grid 결과 1위로 수정
FINAL_N_ESTIMATORS = 1200  # 최종은 항상 1200으로 (탐색은 속도 위해 600 사용했음)

test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
raw_X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)

final_pair = pair_prediction(raw_X, raw_X_test, y)
final_tree = tree_prediction(raw_X, raw_X_test, y, FINAL_N_ESTIMATORS, BEST_MIN_LEAF, BEST_MAX_FEATURES)
final_prediction = np.clip(np.round((1 - PAIR_WEIGHT) * final_tree + PAIR_WEIGHT * final_pair, 2), 0.0, 1.0)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_tree_hparam_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_tree_hparam_0806.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.59
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.50
4,TEST_0004,0.53
5,TEST_0005,0.45
6,TEST_0006,0.78
7,TEST_0007,0.47
8,TEST_0008,0.58
9,TEST_0009,0.88
